# MizanIQ Phase-2 OCR/Vision Benchmark (Kaggle lab)

Independent experiment notebook: load frozen benchmark PNGs, run ONE candidate model,
export machine-readable results in the standard schema, record latency/model/device.

**Setup:** upload `inputs/`, `manifest.json`, `ocr_benchmark_truth.json` from local
`data/benchmark/dataset_v0.1/` (see `notebooks/README.md`). Enable GPU only for
model classes B/C/D. No weights or secrets are stored in this notebook.

In [ ]:
import json
import time
from pathlib import Path

WORKSPACE = Path('.')  # inputs/, manifest.json, ocr_benchmark_truth.json alongside
manifest = json.loads((WORKSPACE / 'manifest.json').read_text(encoding='utf-8'))
truth = json.loads((WORKSPACE / 'ocr_benchmark_truth.json').read_text(encoding='utf-8'))
print(f"loaded {len(manifest['entries'])} benchmark inputs")
for entry in manifest['entries']:
    print(' ', entry['document_id'], entry['role'], entry['tasks'], entry['workspace_file'])

# ---- experiment identity (FILL IN per run) ----
MODEL = 'candidate-name'       # e.g. 'tesseract', 'paddleocr-vl', 'qwen2-vl'
MODEL_VERSION = 'x.y.z'        # exact version / checkpoint id
DEVICE = 'kaggle-T4'           # kaggle-T4 | kaggle-CPU | ...

## Candidate cell (FILL IN on Kaggle)

Implement `run_candidate(image_path)` for exactly one model class.
Return `(text, fields, tables, visual_description)`. Keep `fields` keys aligned with
`ocr_benchmark_truth.json` numeric/identifier names so the local scorer matches directly.

Class-A example (uncomment after `!apt install tesseract-ocr tesseract-ocr-ara` +
`pip install pytesseract pillow`):

```python
# from PIL import Image
# import pytesseract
# def run_candidate(image_path):
#     img = Image.open(image_path)
#     text = pytesseract.image_to_string(img, lang='ara+eng')
#     return text, {}, [], ''
```

In [ ]:
def run_candidate(image_path):
    """Run ONE candidate on a workspace image.

    Returns (text, fields, tables, visual_description).
    Fill in per experiment; leave raising until implemented.
    """
    raise NotImplementedError(
        'implement run_candidate() for exactly one candidate model')


def blank_result(document_id):
    return {
        'model': MODEL,
        'model_version': MODEL_VERSION,
        'device': DEVICE,
        'document_id': document_id,
        'latency_ms': 0.0,
        'text': '',
        'fields': {},
        'tables': [],
        'visual_description': '',
        'warnings': [],
    }

In [ ]:
results = []
for entry in manifest['entries']:
    image_path = WORKSPACE / entry['workspace_file']
    result = blank_result(entry['document_id'])
    started = time.perf_counter()
    try:
        text, fields, tables, visual = run_candidate(image_path)
    except NotImplementedError as exc:
        result['warnings'] = [str(exc)]
    else:
        result['text'] = text
        result['fields'] = fields
        result['tables'] = tables
        result['visual_description'] = visual
    result['latency_ms'] = (time.perf_counter() - started) * 1000.0
    results.append(result)

with open('kaggle_results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"{'document':10} {'ms':>10}  warnings")
for result in results:
    print(f"{result['document_id']:10} {result['latency_ms']:10.0f}  "
          f"{'; '.join(result['warnings'])}")
print('\nsaved kaggle_results.json — download it to local data/benchmark/results/ for scoring')